# 02 — Multi-Hub Visibility & Window Bucketing

Documents the gateway clock offset finding and validates the 20-minute bucketing decision.

**Key finding going in:** Each BioHub has a fixed clock offset baked in at boot time. Across 10 gateways reporting the same device, timestamps spread up to ~9 min 25 sec within a single logical observation window. A 20-minute bucket captures all gateways consistently; a 10-minute bucket splits observations across boundaries.

**Sections:**
1. Gateway clock offset analysis
2. Bucketing comparison: 10 min vs 20 min
3. Vector completeness with 20-minute window

## Section 1 — Connection Setup

Credentials are loaded from a `.env` file in the repo root using `python-dotenv`.
This avoids hardcoding credentials and works reliably across terminal sessions
and VS Code notebook kernels.

**Setup (one time only):**
1. Create a file called `.env` in the `bio-tracker/` root (already gitignored)
2. Add these three lines with no spaces around `=` and no quotes:
```
DATABRICKS_SERVER_HOSTNAME=dbc-116b948a-d644.cloud.databricks.com
DATABRICKS_HTTP_PATH=/sql/1.0/warehouses/d89ac2793b1f6d5c
DATABRICKS_TOKEN=your_token_here
```
3. Install dotenv if needed: `pip install python-dotenv`

**Important:** Cell 1 must run before Cell 2. Always run cells top to bottom
or use "Run All" — never run the connection cell without running the dotenv
cell first.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file BEFORE any other imports
# .env lives in the repo root, one level up from analysis/
load_dotenv("../.env")

# Verify all three variables loaded correctly
print("HOSTNAME:", os.getenv("DATABRICKS_SERVER_HOSTNAME"))
print("HTTP_PATH:", os.getenv("DATABRICKS_HTTP_PATH"))
print("TOKEN set:", os.getenv("DATABRICKS_TOKEN") is not None)

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from databricks import sql

# Add repo root to path so we can import pipeline modules
sys.path.append("..")
from utils.helpers import floor_timestamp
from src.pipeline.wrangler import wrangle
import config

OUTPUT_DIR = "../outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
sns.set_theme(style="darkgrid")

# Device confirmed to be seen by 10 gateways simultaneously
SAMPLE_DEVICE = "t42032c01422c1d01380c1401"

connection = sql.connect(
    server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
    access_token    = os.getenv("DATABRICKS_TOKEN")
)

def run_query(query: str) -> pd.DataFrame:
    """Run a SQL query and return results as a pandas DataFrame."""
    with connection.cursor() as cursor:
        cursor.execute(query)
        return pd.DataFrame([row.asDict() for row in cursor.fetchall()])

print("Connected.")

## Section 1 — Gateway Clock Offset Analysis

Each gateway has a fixed reporting offset from the top of the hour. This section makes that visible.

In [ ]:
# Pull all readings for the sample device over a 4-hour window
# We use a known-good time range where 10 gateways are active
offset_query = f"""
SELECT
    reported_scan_timestamp,
    gateway_id,
    rssi
FROM etl_device_telemetry_bronze.gateway_scan_telemetry
WHERE scanned_device_id = '{SAMPLE_DEVICE}'
  AND reported_scan_timestamp >= '2025-03-04T14:00:00Z'
  AND reported_scan_timestamp <  '2025-03-04T18:00:00Z'
ORDER BY reported_scan_timestamp
"""

offset_df = run_query(offset_query)
offset_df["reported_scan_timestamp"] = pd.to_datetime(offset_df["reported_scan_timestamp"], utc=True)

print(f"Rows: {len(offset_df)}")
print(f"Gateways: {offset_df['gateway_id'].nunique()}")
offset_df.head(10)

In [ ]:
# Plot each gateway's reporting times as a horizontal timeline
# Each gateway should form a clean, evenly-spaced line at its own offset
gateways = offset_df["gateway_id"].unique()
gw_index = {gw: i for i, gw in enumerate(gateways)}

fig, ax = plt.subplots(figsize=(14, 6))
for gw_id, group in offset_df.groupby("gateway_id"):
    ax.scatter(
        group["reported_scan_timestamp"],
        [gw_index[gw_id]] * len(group),
        s=15, alpha=0.8
    )

ax.set_yticks(range(len(gateways)))
ax.set_yticklabels(gateways, fontsize=8)
ax.set_xlabel("Timestamp (UTC)")
ax.set_title(f"Reporting timelines per gateway — device {SAMPLE_DEVICE[:12]}...\nEach row = one gateway; each dot = one report")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}02_gateway_timelines.png", dpi=150)
plt.show()
print("Saved: 02_gateway_timelines.png")

In [ ]:
# For each logical 20-minute window, measure the spread (max - min timestamp)
# This quantifies the clock drift problem
offset_df["window_20min"] = offset_df["reported_scan_timestamp"].apply(
    lambda ts: floor_timestamp(ts, 20)
)

spread = offset_df.groupby("window_20min")["reported_scan_timestamp"].agg(
    spread_seconds=lambda x: (x.max() - x.min()).total_seconds(),
    gateways_in_window="count"
).reset_index()

print("Timestamp spread within each 20-minute window:")
print(spread[["spread_seconds", "gateways_in_window"]].describe().round(1))

## Section 2 — Bucketing Comparison: 10 min vs 20 min

With a 10-minute floor, the ~9.5 minute spread means some gateways fall into the wrong bucket. Here we quantify the improvement from switching to 20 minutes.

In [ ]:
# Apply both floor sizes and count gateways per window
offset_df["window_10min"] = offset_df["reported_scan_timestamp"].apply(
    lambda ts: floor_timestamp(ts, 10)
)
offset_df["window_20min"] = offset_df["reported_scan_timestamp"].apply(
    lambda ts: floor_timestamp(ts, 20)
)

# Count distinct gateways per window under each strategy
gw_per_window_10 = offset_df.groupby("window_10min")["gateway_id"].nunique()
gw_per_window_20 = offset_df.groupby("window_20min")["gateway_id"].nunique()

print("=== 10-minute bucketing ===")
print(gw_per_window_10.value_counts().sort_index())
print(f"\nWindows with <5 gateways: {(gw_per_window_10 < 5).sum()} / {len(gw_per_window_10)}")

print("\n=== 20-minute bucketing ===")
print(gw_per_window_20.value_counts().sort_index())
print(f"\nWindows with <5 gateways: {(gw_per_window_20 < 5).sum()} / {len(gw_per_window_20)}")

In [ ]:
# Side-by-side histogram comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

for ax, data, title in zip(
    axes,
    [gw_per_window_10, gw_per_window_20],
    ["10-minute bucketing", "20-minute bucketing"]
):
    counts = data.value_counts().sort_index()
    ax.bar(counts.index, counts.values, color="steelblue", edgecolor="white")
    ax.set_xlabel("Gateways captured in window")
    ax.set_ylabel("Number of windows")
    ax.set_title(title)
    ax.set_xticks(counts.index)

fig.suptitle("Effect of bucket size on multi-gateway vector completeness", fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}02_bucketing_comparison.png", dpi=150)
plt.show()
print("Saved: 02_bucketing_comparison.png")

## Section 3 — Vector Completeness with 20-Minute Window

Run the real wrangler on a sample of deployment data and measure how complete the resulting RSSI vectors are.

In [ ]:
# Pull a larger real data sample for wrangler validation
# Using one day of data from a gateway with high coverage
sample_query = """
SELECT
    gateway_id,
    reported_scan_timestamp AS timestamp,
    scanned_device_id       AS device_id,
    rssi
FROM etl_device_telemetry_bronze.gateway_scan_telemetry
WHERE gateway_id != 'bio_id_test_scan_results'
  AND reported_scan_timestamp >= '2025-03-04T00:00:00Z'
  AND reported_scan_timestamp <  '2025-03-05T00:00:00Z'
  AND rssi <= 0
  AND rssi >= -100
"""

sample_df = run_query(sample_query)
sample_df["timestamp"] = pd.to_datetime(sample_df["timestamp"], utc=True)
sample_df["rssi"] = sample_df["rssi"].astype(float)

print(f"Sample rows: {len(sample_df):,}")
print(f"Devices: {sample_df['device_id'].nunique()}")
print(f"Gateways: {sample_df['gateway_id'].nunique()}")
sample_df.head()

In [ ]:
# Run the wrangler — this pivots long format into RSSI vectors
# wrangle() uses WINDOW_MINUTES and MIN_HUBS_PER_WINDOW from config.py
wide_df = wrangle(sample_df)

print(f"\nWide format shape: {wide_df.shape}")
print(f"Columns (hubs): {list(wide_df.columns)}")
wide_df.head()

In [ ]:
# Measure vector completeness
# non_null_count = number of hubs that saw a device in a given window
hub_counts = wide_df.notna().sum(axis=1)

print("Hub readings per vector (after MIN_HUBS_PER_WINDOW filter):")
print(hub_counts.value_counts().sort_index())
print(f"\nMean hubs per vector: {hub_counts.mean():.2f}")
print(f"Vectors with 2+ hubs: {(hub_counts >= 2).mean()*100:.1f}%")
print(f"Vectors with 5+ hubs: {(hub_counts >= 5).mean()*100:.1f}%")

# Bar chart of hub count distribution in final vectors
fig, ax = plt.subplots(figsize=(8, 4))
vc = hub_counts.value_counts().sort_index()
ax.bar(vc.index, vc.values, color="steelblue")
ax.set_xlabel("Hubs per RSSI vector")
ax.set_ylabel("Number of vectors")
ax.set_title("RSSI vector completeness after 20-min bucketing + wrangler")
ax.set_xticks(vc.index)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}02_vector_completeness.png", dpi=150)
plt.show()
print("Saved: 02_vector_completeness.png")

In [ ]:
connection.close()
print("Connection closed.")